In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/duongquanganh/raw-dataset/vlsp2018_restaurant/2-VLSP2018-SA-Restaurant-dev.txt
/kaggle/input/datasets/duongquanganh/raw-dataset/vlsp2018_restaurant/3-VLSP2018-SA-Restaurant-test.txt
/kaggle/input/datasets/duongquanganh/raw-dataset/vlsp2018_restaurant/2-VLSP2018-SA-Restaurant-dev.csv
/kaggle/input/datasets/duongquanganh/raw-dataset/vlsp2018_restaurant/1-VLSP2018-SA-Restaurant-train.csv
/kaggle/input/datasets/duongquanganh/raw-dataset/vlsp2018_restaurant/3-VLSP2018-SA-Restaurant-test.csv
/kaggle/input/datasets/duongquanganh/raw-dataset/vlsp2018_restaurant/1-VLSP2018-SA-Restaurant-train.txt


In [2]:
import re
from pathlib import Path


def load_reviews_and_labels(txt_path):
	review_list = []
	label_list = []

	review_lines = []

	with Path(txt_path).open("r", encoding="utf-8") as f:
		for raw_line in f:
			line = raw_line.strip()

			# Each sample starts with #1, #2, ...
			if re.fullmatch(r"#\d+", line):
				if review_lines:
					review_list.append(" ".join(review_lines).strip())
					review_lines = []
				continue

			# Label line: {ASPECT, sentiment}, {...}
			if line.startswith("{"):
				labels = re.findall(r"\{\s*([^,{}]+)\s*,\s*([^{}]+?)\s*\}", line)
				label_list.append([(a.strip(), s.strip()) for a, s in labels])
			elif line:
				review_lines.append(line)

	# Last sample review
	if review_lines:
		review_list.append(" ".join(review_lines).strip())

	return review_list, label_list

In [3]:
train_txt = "/kaggle/input/datasets/duongquanganh/raw-dataset/vlsp2018_restaurant/1-VLSP2018-SA-Restaurant-train.txt"
val_txt = "/kaggle/input/datasets/duongquanganh/raw-dataset/vlsp2018_restaurant/2-VLSP2018-SA-Restaurant-dev.txt"
test_txt = "/kaggle/input/datasets/duongquanganh/raw-dataset/vlsp2018_restaurant/3-VLSP2018-SA-Restaurant-test.txt"

In [4]:
train_review_list, train_label_list = load_reviews_and_labels(train_txt)
print(f"Loaded {len(train_review_list)} samples of train")

val_review_list, val_label_list = load_reviews_and_labels(val_txt)
print(f"Loaded {len(val_review_list)} samples of val")

test_review_list, test_label_list = load_reviews_and_labels(test_txt)
print(f"Loaded {len(test_review_list)} samples of test")

Loaded 2961 samples of train
Loaded 1290 samples of val
Loaded 500 samples of test


In [5]:
# 1. Install dependencies (zstd is required for extraction)
!apt-get update && apt-get install -y zstd

# 2. Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Start two independent Ollama servers, one per T4 GPU
import subprocess
import time
import os

if os.path.exists("/usr/local/bin/ollama") or os.path.exists("/usr/bin/ollama"):
    print("Ollama binary found! Starting two independent servers, one per T4 GPU...")

    base_env = os.environ.copy()
    base_env["OLLAMA_NUM_GPU"]      = "99"  # use all layers on whichever GPU is visible
    base_env["OLLAMA_NUM_PARALLEL"] = "2"   # 2 parallel slots per instance (fits in ~14.8 GiB RAM)

    # --- Instance 0: GPU 0, default port 11434 ---
    env0 = base_env.copy()
    env0["CUDA_VISIBLE_DEVICES"] = "0"
    env0["OLLAMA_HOST"]          = "0.0.0.0:11434"
    with open("ollama_gpu0.log", "w") as f:
        subprocess.Popen(["ollama", "serve"], stdout=f, stderr=f, env=env0)

    # --- Instance 1: GPU 1, port 11435 ---
    env1 = base_env.copy()
    env1["CUDA_VISIBLE_DEVICES"] = "1"
    env1["OLLAMA_HOST"]          = "0.0.0.0:11435"
    with open("ollama_gpu1.log", "w") as f:
        subprocess.Popen(["ollama", "serve"], stdout=f, stderr=f, env=env1)

    # Give both servers time to initialize

    import urllib.request

    def wait_for_ollama(host, timeout=60):
        url = f"{host}/api/tags"
        for _ in range(timeout):
            try:
                urllib.request.urlopen(url, timeout=1)
                print(f"{host} is ready.")
                return
            except:
                time.sleep(1)
        raise RuntimeError(f"{host} did not start within {timeout}s")

    wait_for_ollama("http://localhost:11434")
    wait_for_ollama("http://localhost:11435")

    print("Both Ollama servers are ready (GPU-0 → :11434, GPU-1 → :11435).")
else:
    print("Installation failed. Check the output above for errors.")


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 https://cli.github.com/packages stable InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,943 kB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,302 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:

In [6]:
# Pull the model on both Ollama instances (each instance has its own model store)
!OLLAMA_HOST=http://localhost:11434 ollama pull qwen2.5:14b-instruct
!OLLAMA_HOST=http://localhost:11435 ollama pull qwen2.5:14b-instruct

]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 2049f5674b1e:   1% ▕                  ▏  56 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   1% ▕                  ▏ 101 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   2% ▕                  ▏ 194 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   3% ▕                  ▏ 299 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   4% ▕                  ▏ 341 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   5% ▕                  ▏ 445 MB/9.0 GB                  pulling manifest 
pulling 2049f5674b1e:   6% ▕█  

In [7]:
!pip install ollama

In [8]:
import ollama

# Let's test the 14B model
try:
    response = ollama.chat(model='qwen2.5:14b-instruct', messages=[
        {'role': 'user', 'content': 'Explain the concept of "quantization" in LLMs like a five-year-old.'}
    ])
    print("--- Response from Qwen 2.5 14B ---")
    print(response['message']['content'])
except Exception as e:
    print(f"An error occurred: {e}")
    print("Tip: If it says 'connection refused', the background server might have crashed due to VRAM limits.")

--- Response from Qwen 2.5 14B ---
Okay, let's imagine you have a big box full of watercolors with all sorts of colors. Now, when you're painting, you can mix these colors to make even more shades and tones - almost any color your imagination wants! But sometimes, you might want to share your painting or send it to a friend through the mail, but only certain colors are allowed in the box that goes out.

Quantization is like choosing just a few special watercolors from all those many choices. In big computer models for talking and understanding language (like LLMs), quantization means picking simpler numbers that take up less space and use less energy when the computer does its work, but still try to keep as much of the information as possible.

So instead of having every single shade and tone available like in your big box of watercolors, you might only pick red, yellow, blue, and green. This makes it easier to send your painting through the mail or share it with a friend quickly, even

In [9]:
import json

# One persistent client per server instance keeps connections warm
_clients = {
    "http://localhost:11434": ollama.Client(host="http://localhost:11434"),
    "http://localhost:11435": ollama.Client(host="http://localhost:11435"),
}


def judge(review, host="http://localhost:11434"):
    system_instruction = (
        "Bạn là một trợ lý làm sạch văn bản chuyên cho review nhà hàng bằng tiếng Việt. "
        "Nhiệm vụ của bạn là chuẩn hóa câu review, chỉ chỉnh sửa ở mức hình thức và không làm thay đổi nội dung cốt lõi. "
        "Bạn PHẢI trả về đúng một đối tượng JSON nguyên bản, không kèm bất kỳ văn bản nào khác."
    )

    prompt = f"""
### ĐẦU VÀO
- review: {review}

### MỤC TIÊU
Làm sạch câu review theo các quy tắc sau:
1. Sửa lỗi chính tả.
2. Xóa hoàn toàn HTML tags.
3. Xóa hoàn toàn URL.
4. Xóa hoàn toàn Email.
5. Xóa hoàn toàn số điện thoại.
6. Xóa hoàn toàn hashtag.
7. Xóa hoàn toàn emoji.
8. Xóa các ký tự rác/không cần thiết.

### RÀNG BUỘC QUAN TRỌNG
- Chỉ làm sạch, không thêm thông tin mới.
- Không đổi nghĩa câu.
- Nội dung sau làm sạch vẫn phải giữ nguyên ngữ nghĩa cốt lõi của review gốc.

### YÊU CẦU ĐẦU RA (JSON ONLY)
Trả về đúng định dạng:
{{
  "cleaned_review": "..."
}}
"""

    response = _clients[host].chat(
        model='qwen2.5:14b-instruct',
        messages=[
            {'role': 'system', 'content': system_instruction},
            {'role': 'user', 'content': prompt}
        ],
        format='json',
        options={
            "temperature": 0.0,
            "top_p": 0.1,
        }
    )

    try:
        return json.loads(response.message.content)
    except Exception:
        # Fallback to a valid minimal dict if model output is malformed JSON.
        return {"cleaned_review": str(response.message.content).strip()}

In [10]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

# T4x2 setup: 2 Ollama instances, each OLLAMA_NUM_PARALLEL=2 => 4 concurrent slots.
NUM_WORKERS = 4
OLLAMA_HOSTS = ["http://localhost:11434", "http://localhost:11435"]


def process_review_list(review_list, dataset_name="dataset", num_workers=NUM_WORKERS, hosts=OLLAMA_HOSTS):
    """Clean one review list with multi-threading across multiple Ollama hosts."""
    thread_info = {}
    map_lock = threading.Lock()
    worker_counter = 0

    def get_worker_info():
        """Assign a stable worker label and host to each thread (round-robin by thread)."""
        nonlocal worker_counter
        tid = threading.current_thread().name
        with map_lock:
            if tid not in thread_info:
                slot = worker_counter
                worker_counter += 1
                host = hosts[slot % len(hosts)]
                thread_info[tid] = (f"W-{slot} (GPU-{slot % len(hosts)})", host)
            return thread_info[tid]

    def log(worker, msg):
        ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
        print(f"[{ts}] [{dataset_name}] [{worker}] {msg}", flush=True)

    def process_item(idx, review):
        """Process one review on the assigned GPU host."""
        worker, host = get_worker_info()
        t0 = time.time()

        try:
            result = judge(review=review, host=host)
            cleaned_review = result.get("cleaned_review", "") if isinstance(result, dict) else ""
            error = None
        except Exception as e:
            result = {"cleaned_review": "", "error": str(e)}
            cleaned_review = ""
            error = str(e)

        elapsed = time.time() - t0
        log(worker, f"done sample={idx + 1} ({elapsed:.2f}s)")

        return idx, {
            "index": idx + 1,
            "review": review,
            "cleaned_review": cleaned_review,
            "raw_result": result,
            "error": error,
            "host": host,
            "elapsed_sec": elapsed,
            "dataset": dataset_name,
        }

    subset = list(review_list)
    processed_rows = [None] * len(subset)

    print(
        f"Processing {len(subset)} samples for [{dataset_name}] with {num_workers} workers "
        f"across {len(hosts)} Ollama hosts...\n"
    )

    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futures = {
            pool.submit(process_item, i, review): i
            for i, review in enumerate(subset)
        }

        completed = 0
        for future in as_completed(futures):
            idx, row = future.result()
            processed_rows[idx] = row
            completed += 1

            if completed % 50 == 0 or completed == len(subset):
                print(
                    f">>> [{dataset_name}] progress: {completed}/{len(subset)}",
                    flush=True
                )

    processed_df = pd.DataFrame(processed_rows)
    print(f"\n[{dataset_name}] done. Total processed: {len(processed_df)}")
    return processed_df

In [11]:
processed_train = process_review_list(train_review_list, dataset_name="train")
processed_val = process_review_list(val_review_list, dataset_name="val")
processed_test = process_review_list(test_review_list, dataset_name="test")

Processing 2961 samples for [train] with 4 workers across 2 Ollama hosts...

[12:12:44.368] [train] [W-2 (GPU-0)] done sample=3 (14.06s)
[12:12:44.440] [train] [W-0 (GPU-0)] done sample=1 (14.14s)
[12:12:50.763] [train] [W-3 (GPU-1)] done sample=4 (20.45s)
[12:12:52.036] [train] [W-1 (GPU-1)] done sample=2 (21.73s)
[12:12:57.826] [train] [W-1 (GPU-1)] done sample=8 (5.79s)
[12:13:00.338] [train] [W-0 (GPU-0)] done sample=6 (15.90s)
[12:13:03.314] [train] [W-3 (GPU-1)] done sample=7 (12.55s)
[12:13:05.860] [train] [W-3 (GPU-1)] done sample=11 (2.55s)
[12:13:07.158] [train] [W-2 (GPU-0)] done sample=5 (22.79s)
[12:13:08.207] [train] [W-1 (GPU-1)] done sample=9 (10.38s)
[12:13:18.235] [train] [W-3 (GPU-1)] done sample=12 (12.37s)
[12:13:25.361] [train] [W-0 (GPU-0)] done sample=10 (25.02s)
[12:13:26.048] [train] [W-1 (GPU-1)] done sample=14 (17.84s)
[12:13:27.223] [train] [W-3 (GPU-1)] done sample=15 (8.99s)
[12:13:42.920] [train] [W-2 (GPU-0)] done sample=13 (35.76s)
[12:13:43.944] [trai

In [12]:
processed_train.to_csv("processed_train.csv", index=False, encoding="utf-8-sig")
print("Saved: processed_train.csv")
processed_val.to_csv("processed_val.csv", index=False, encoding="utf-8-sig")
print("Saved: processed_val.csv")
processed_test.to_csv("processed_test.csv", index=False, encoding="utf-8-sig")
print("Saved: processed_test.csv")

Saved: processed_train.csv
Saved: processed_val.csv
Saved: processed_test.csv
